<a href="https://colab.research.google.com/github/nisaerdemli/imageprocessing/blob/main/hafta6g%C3%B6rev5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import numpy as np

# 1. Resmi yükleme ve ön işleme (Gri tonlama, Median filtre, Canny kenar)
img = cv2.imread('indir.jpg')
if img is None:
  print('Resim yüklenemedi. Lütfen dosya adını kontrol edin.')
else:
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # Gürültüyü azaltmak için median filtre
  median_gray = cv2.medianBlur(gray, 5)

  # Canny kenar tespiti
  edges = cv2.Canny(median_gray, 50, 150)
  median_edges = cv2.medianBlur(edges, 3)

  height, width = median_edges.shape

  # 2. Açı ve Yarıçap Parametreleri & Sin/Cos Ön Hesaplama (Lookup Table)
  thetas = np.deg2rad(np.arange(0, 360, 5))
  num_thetas = len(thetas)

  # Sin ve Cos değerlerini baştan hesaplayıp bellekte tutma
  cos_thetas = np.cos(thetas)
  sin_thetas = np.sin(thetas)

  # Aranacak yarıçap aralığı
  min_radius = 10
  max_radius = 90
  radius_step = 2
  radii = np.arange(min_radius, max_radius, radius_step)
  num_radii = len(radii)

  # 3. 3 Boyutlu Oylama Matrisini Oluşturma [Yükseklik (b), Genişlik (a), Yarıçap (r)]
  accumulator = np.zeros((height, width, num_radii), dtype=np.uint64)

  # Kenar piksel koordinatlarını alma
  edge_y, edge_x = np.nonzero(median_edges)

  # 4. Oylama İşleminin Açıkça Gösterilmesi (Voting Process)
  print('3 Boyutlu Hough Çember oylama işlemi gerçekleştiriliyor...')
  for i in range(len(edge_x)):
    x = edge_x[i]
    y = edge_y[i]

    # Her bir kenar pikseli için yarıçaplar ve açılar üzerinden oylama
    for r_idx, r in enumerate(radii):
      for t_idx in range(num_thetas):
        # Parametrik çember denklemi ile merkez (a, b) hesaplanır:
        a = int(x - r * cos_thetas[t_idx])
        b = int(y - r * sin_thetas[t_idx])

        # Merkez görüntü sınırları içindeyse akümülatör ilgili hücre artırılır
        if 0 <= a < width and 0 <= b < height:
          accumulator[b, a, r_idx] += 1

  # 5. Eş Merkezli Çemberlerin Doğrudan Tespiti
  # Eş merkezli çemberler, aynı (a, b) merkez noktasında birden fazla farklı yarıçapta (r)
  # yeterli oyu alan yapılardır.
  threshold = 20  # Her bir çember katmanı için minimum oy eşiği
  min_concentric_layers = (
      2  # Aynı merkezde en az kaç farklı yarıçap olmalı?
  )

  img_concentric = img.copy()
  detected_concentric_sets = []

  # Oylama matrisini merkez (b, a) bazında tarayarak eş merkezli yapıları buluyoruz
  for b in range(height):
    for a in range(width):
      # Belirli bir merkezin tüm yarıçaplardaki oyları
      votes_at_center = accumulator[b, a, :]
      # Eşik değerini aşan yarıçap indisleri
      valid_r_indices = np.where(votes_at_center >= threshold)[0]

      # Eğer bu merkezde birden fazla geçerli yarıçap varsa, bunlar eş merkezlidir!
      if len(valid_r_indices) >= min_concentric_layers:
        valid_radii = [radii[r_idx] for r_idx in valid_r_indices]
        detected_concentric_sets.append(((a, b), valid_radii))

  # 6. Bulunan Eş Merkezli Çemberleri Orijinal Resim Üzerinde Çizme
  print(
      f'Toplam {len(detected_concentric_sets)} adet eş merkezli merkez'
      ' tespit edildi.'
  )

  for center, radii_list in detected_concentric_sets:
    cx, cy = center
    # Merkez noktasını kırmızı renkle işaretle (BGR: 0, 0, 255)
    cv2.circle(img_concentric, (cx, cy), 3, (0, 0, 255), -1)

    # Aynı merkeze ait tüm eş merkezli çemberlerin çevrelerini mavi renkle çiz (BGR: 255, 0, 0)
    for r in radii_list:
      cv2.circle(img_concentric, (cx, cy), r, (255, 0, 0), 2)

  # Sonuçları Colab Üzerinde Görselleştirme
  print('Canny + Median Kenar Görüntüsü:')
  cv2_imshow(median_edges)

  print('Eş Merkezli Çember Tespiti Sonucu (Mavi Çemberler, Kırmızı Merkezler):')
  cv2_imshow(img_concentric)